# NLS candidate-action scoring — completed curves

This notebook reads the canonical histories produced by `merge_downloaded_continuations.py`. Base and continuation W&B runs remain separate online, while the local curve is reconstructed on the original environment-step axis. At a continuation boundary, continuation data replaces any overlapping tail in the base run.

The training-curve analysis works before all full-test evaluations are available; the last section adds each full-test result automatically when its JSON file appears locally.

In [ ]:
from pathlib import Path
import json, re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 80)

def find_task_dir(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'main.py').is_file():
            return candidate
        nested = candidate / 'Topology_Task'
        if (nested / 'main.py').is_file():
            return nested
    raise RuntimeError('Could not locate Topology_Task')

TASK_DIR = find_task_dir()
GROUP_DIR = TASK_DIR / 'outputs' / 'run_data' / 'NLS_cas_hl'
MERGED_ROOT = GROUP_DIR / 'merged_runs'
BOUNDARY_PATH = GROUP_DIR / 'continuation_boundaries.csv'
FIG_DIR = TASK_DIR.parent / 'latex' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_RE = re.compile(r'^cas_hl_NLS_(mean|tmean)_(f[01])_(a0h[01])_s0$')
METRIC_CANDIDATES = [
    'test/charts/episodic_survival',
    'test/episodic_survival',
]
SMOOTH_WINDOW = 7
print('task:', TASK_DIR)
print('canonical histories:', MERGED_ROOT)

## Load and validate the reconstructed histories

In [ ]:
manifest_path = GROUP_DIR / 'merged_manifest.csv'
if not manifest_path.exists():
    raise FileNotFoundError(
        f'Missing {manifest_path}. Run scripts/merge_downloaded_continuations.py first.'
    )
manifest = pd.read_csv(manifest_path)
boundaries = pd.read_csv(BOUNDARY_PATH) if BOUNDARY_PATH.exists() and BOUNDARY_PATH.stat().st_size else pd.DataFrame()

curves = {}
coverage_rows = []
for row in manifest.itertuples(index=False):
    match = RUN_RE.fullmatch(str(row.name))
    if match is None:
        continue
    history = pd.read_parquet(row.history_parquet)
    metric = next((key for key in METRIC_CANDIDATES if key in history and history[key].notna().any()), None)
    if metric is None:
        coverage_rows.append({'run_name': row.name, 'status': 'missing survival metric'})
        continue
    curve = history[['_step', metric, 'history_segment']].dropna(subset=[metric]).copy()
    curve['_step'] = pd.to_numeric(curve['_step'], errors='coerce')
    curve['survival_percent'] = 100 * pd.to_numeric(curve[metric], errors='coerce')
    curve = curve.dropna(subset=['_step', 'survival_percent']).sort_values('_step')
    curve = curve.groupby('_step', as_index=False).last()
    pool, features, head = match.groups()
    label = f'{pool}/{features}/{head}'
    curves[row.name] = {
        'label': label, 'pool': pool, 'features': features, 'a0_head': head,
        'metric': metric, 'data': curve, 'segments': int(row.segments),
    }
    coverage_rows.append({
        'run_name': row.name, 'label': label, 'segments': int(row.segments),
        'test_points': len(curve), 'first_test_step': int(curve._step.min()),
        'last_test_step': int(curve._step.max()), 'status': 'ok',
    })

coverage = pd.DataFrame(coverage_rows).sort_values('run_name')
display(coverage)
if len(curves) != 8:
    print(f'WARNING: expected 8 reconstructed cells, found {len(curves)}')
if not boundaries.empty:
    display(boundaries.sort_values(['base_run_name', 'start_step']))

## Periodic test survival during training

Dotted vertical lines mark the first logged step of a separate continuation run. They are provenance markers, not algorithmic events.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.4))
palette = plt.get_cmap('tab10')
colors = {}
for index, (run_name, item) in enumerate(sorted(curves.items(), key=lambda pair: pair[1]['label'])):
    curve = item['data']
    smooth = curve['survival_percent'].rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean()
    color = palette(index)
    colors[run_name] = color
    ax.plot(curve['_step'] / 1e6, smooth, lw=1.7, color=color, label=item['label'])

if not boundaries.empty:
    for boundary in boundaries.itertuples(index=False):
        if boundary.base_run_name in colors:
            ax.axvline(boundary.start_step / 1e6, color=colors[boundary.base_run_name], ls=':', lw=1.1, alpha=0.9)

ax.set(xlabel='environment steps (millions)', ylabel='periodic test survival (%)', ylim=(0, 104))
ax.grid(alpha=0.25)
ax.legend(fontsize=8.5, ncol=2, loc='lower right', framealpha=0.92)
ax.set_title('NLS candidate-action scoring — reconstructed seed-0 curves')
fig.tight_layout()
figure_path = FIG_DIR / 'nls_cas_hl_completed_curves.png'
fig.savefig(figure_path, dpi=220)
print('saved:', figure_path)
plt.show()

## Training-curve summary

The maximum periodic evaluation is useful for diagnosing learning, but it is not a substitute for the 201-chronic full-test evaluation of the saved best checkpoint.

In [ ]:
summary_rows = []
for run_name, item in curves.items():
    curve = item['data']
    best_index = curve['survival_percent'].idxmax()
    final_window = curve.tail(min(5, len(curve)))
    summary_rows.append({
        'run_name': run_name, 'cell': item['label'], 'pool': item['pool'],
        'features': item['features'], 'a0_head': item['a0_head'],
        'segments': item['segments'], 'test_points': len(curve),
        'best_periodic_survival': curve.loc[best_index, 'survival_percent'],
        'best_periodic_step': int(curve.loc[best_index, '_step']),
        'last_periodic_survival': curve.iloc[-1]['survival_percent'],
        'last5_periodic_mean': final_window['survival_percent'].mean(),
        'last_test_step': int(curve.iloc[-1]['_step']),
    })
training_summary = pd.DataFrame(summary_rows).sort_values('best_periodic_survival', ascending=False).reset_index(drop=True)
display(training_summary.round(3))

## Full-test results when available

This cell searches recursively under `outputs/full_test_eval`. Missing cells remain explicit instead of being replaced by periodic ten-episode evaluations.

In [ ]:
full_test_rows = []
for path in sorted((TASK_DIR / 'outputs' / 'full_test_eval').rglob('*.json')):
    match = re.search(r'(cas_hl_NLS_(?:mean|tmean)_f[01]_a0h[01]_s0)', path.name)
    if match is None:
        continue
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    survival = payload.get('survival_percent')
    if survival is None and payload.get('survival_frac') is not None:
        survival = 100 * float(payload['survival_frac'])
    if survival is None:
        continue
    full_test_rows.append({
        'run_name': match.group(1), 'full_test_survival': float(survival),
        'checkpoint_step': payload.get('checkpoint_global_step'),
        'eval_episodes': payload.get('eval_episodes'),
        'created_at': payload.get('created_at', ''), 'result_path': str(path),
    })

if full_test_rows:
    full_test = pd.DataFrame(full_test_rows).sort_values('created_at').drop_duplicates('run_name', keep='last')
else:
    full_test = pd.DataFrame(columns=['run_name', 'full_test_survival', 'checkpoint_step', 'eval_episodes', 'result_path'])
results = training_summary.merge(full_test, on='run_name', how='left')
results['full_test_status'] = np.where(results['full_test_survival'].notna(), 'available', 'pending')
results = results.sort_values(['full_test_status', 'full_test_survival', 'best_periodic_survival'], ascending=[True, False, False])
display(results[[
    'cell', 'segments', 'best_periodic_survival', 'best_periodic_step',
    'last5_periodic_mean', 'full_test_survival', 'checkpoint_step',
    'eval_episodes', 'full_test_status'
]].round(3))
print(f"full-test coverage: {results['full_test_survival'].notna().sum()} / {len(results)}")